# WelfareOne 통합 RAG·RAGAS 평가 (논문 초안용)

이 노트북은 한 번의 실행 흐름에서 다음을 평가합니다.

1. **파이프라인 평가**: 정책 식별, 질문 의도 분류, 섹션 선택, 원문 청크 검색
2. **RAGAS 평가**: Faithfulness, Context Precision, Context Recall, Factual Correctness
3. **논문용 결과 저장**: 문항별 결과, 전체 평균, 질문 유형·의도별 요약

### 실행 방법
- 처음에는 `PIPELINE_LIMIT = 3`, `RAGAS_LIMIT = 3`으로 끝까지 시험합니다.
- 정상 작동하면 두 값을 `None`으로 바꾸고 다시 실행합니다.
- `region`과 `age`는 서비스에서 별도로 받은 구조화 정보로 사용하고, 자연어 질문은 검색·분류에 사용합니다.
- RAGAS 결과는 문항마다 체크포인트 CSV에 저장되므로 중단 후 다시 실행할 수 있습니다.


In [ ]:
# 최초 1회만 주석을 해제해 설치하세요.
# %pip install "ragas>=0.4,<0.5" datasets openai google-genai chromadb sentence-transformers openpyxl


In [1]:
from pathlib import Path
import json
import pandas as pd

PROJECT_DIR = Path(r"C:\ai_project01\WelfareOne")

EVAL_FILE = Path(r"C:\Users\user\Desktop\인공지능-권혜윤교수님\05.프로젝트\outputs\WelfareOne_RAGAS_평가데이터_30문항.xlsx")

eval_df = pd.read_excel(
    EVAL_FILE,
    sheet_name="RAGAS_평가작성",
    header=3,  # 엑셀의 4번째 행이 열 이름
)

# 완전히 빈 행 제거
eval_df = eval_df.dropna(
    subset=["id", "user_input", "expected_policy_id"]
).copy()

# 나이를 25.0이 아닌 25로 관리
eval_df["age"] = pd.to_numeric(
    eval_df["age"],
    errors="coerce",
).astype("Int64")

# expected_sections를 문자열에서 파이썬 리스트로 변환
def parse_json_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value) or not str(value).strip():
        return []

    return json.loads(str(value))


eval_df["expected_sections"] = (
    eval_df["expected_sections"].apply(parse_json_list)
)

print("평가 문항 수:", len(eval_df))
print("열 목록:", eval_df.columns.tolist())

display(
    eval_df[
        [
            "id",
            "user_input",
            "expected_policy_name",
            "expected_intent",
            "expected_sections",
            "reference",
        ]
    ].head()
)

# 평가 데이터 기본 검증
required_columns = [
    "id", "region", "question_type", "expected_policy_id",
    "expected_policy_name", "user_input", "expected_intent",
    "expected_sections", "reference", "source",
]
missing_columns = [c for c in required_columns if c not in eval_df.columns]
if missing_columns:
    raise ValueError(f"필수 열 누락: {missing_columns}")
if eval_df["id"].duplicated().any():
    raise ValueError("중복된 평가 ID가 있습니다.")

print("질문 유형 분포:")
display(eval_df["question_type"].value_counts(dropna=False).rename("문항 수"))
print("의도 분포:")
display(eval_df["expected_intent"].value_counts(dropna=False).rename("문항 수"))

# 서식·링크 질문인데 reference에 URL이 없으면 검수 대상으로 표시
link_review = eval_df[
    eval_df["expected_intent"].isin(["required_documents", "application"])
    & eval_df["user_input"].astype(str).str.contains("서식|링크|사이트|온라인", regex=True)
    & ~eval_df["reference"].astype(str).str.contains("http://|https://", regex=True)
]
if len(link_review):
    print("링크 포함 여부를 검토할 문항:", link_review["id"].tolist())


평가 문항 수: 30
열 목록: ['id', 'region', 'age', 'question_type', 'expected_policy_id', 'expected_policy_name', 'user_input', 'expected_intent', 'expected_sections', 'reference', 'reference_contexts', 'reference_context_ids', 'retrieved_contexts', 'retrieved_context_ids', 'response', 'source', 'author']


,id,user_input,expected_policy_name,expected_intent,expected_sections,reference
0,RAGAS01,장애인보장구수리지원사업은 얼마 정도 지원받을 수 있나요?,장애인보장구수리지원사업,benefit,[benefit],예산 범위 내에서 부품비용을 1인당 연간 최대 20만 원까지 소득에 따라 차등 지원...
1,RAGAS02,국가보훈대상자가 받는 보훈영예수당은 얼마나 지급되나요?,보훈영예수당,benefit,[benefit],보훈영예수당은 월 25만 원씩 지급합니다.
2,RAGAS03,25살 직장인인데 청년 중개보수 및 이사비 지원을 받을 수 있는 기준이 무엇인가요?,부산 청년 중개보수 및 이사비 지원,eligibility,[target_criteria],18~39세로 부산시에 전입했거나 부산 시내에서 이사 후 전입신고한 일하는 청년이 ...
3,RAGAS04,24살 미취업 청년입니다. 청년 구직활동비를 지원받으려면 어떤 자격을 갖춰야 하나요?,해운대구 청년 구직활동비 지원 사업,eligibility,[target_criteria],공고일 기준 19세 이상 39세 이하이며 해운대구에 주민등록을 둔 미취업 청년이어야...
4,RAGAS05,임산부 태아기형아 검사 쿠폰 지원은 어떤 정책인가요?,임산부 태아기형아 검사 쿠폰 지원,policy_overview,"[overview, target_criteria, benefit]",태아 기형의 조기 발견과 출산가정의 경제적 부담 완화를 위한 정책입니다. 김천시보건...


질문 유형 분포:


question_type
직접형    18
간접형    12
Name: 문항 수, dtype: int64

의도 분포:


expected_intent
benefit               11
eligibility            7
application            6
policy_overview        3
required_documents     2
contact                1
Name: 문항 수, dtype: int64

링크 포함 여부를 검토할 문항: ['RAGAS15', 'RAGAS19']


In [2]:
import chromadb
from sentence_transformers import SentenceTransformer

DB_PATH = PROJECT_DIR / "data" / "chroma"
COLLECTION_NAME = "welfare_policy_sections_qwen_v3"
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"

client = chromadb.PersistentClient(
    path=str(DB_PATH)
)

collection = client.get_collection(
    name=COLLECTION_NAME
)

embedder = SentenceTransformer(
    EMBEDDING_MODEL,
    local_files_only=True,
)

print("ChromaDB 위치:", DB_PATH)
print("컬렉션:", COLLECTION_NAME)
print("전체 청크 수:", collection.count())
print("임베딩 모델:", EMBEDDING_MODEL)
print("사용 장치:", embedder.device)

c:\Users\user\anaconda3\envs\welfareone\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 714.19it/s]


ChromaDB 위치: C:\ai_project01\WelfareOne\data\chroma
컬렉션: welfare_policy_sections_qwen_v3
전체 청크 수: 11850
임베딩 모델: Qwen/Qwen3-Embedding-0.6B
사용 장치: cuda:0


In [3]:
from getpass import getpass
import json
import re

from google import genai
from google.genai import types

GEMINI_MODEL = "gemini-3.5-flash-lite"

api_key = getpass("Gemini API 키: ")
gemini = genai.Client(api_key=api_key)


def normalize_text(text):
    """공백 차이를 제거해 정책명을 비교한다."""
    return re.sub(r"\s+", "", str(text or "")).strip()


# 정책마다 하나씩 있는 candidate_summary를 정책 목록으로 사용
policy_result = collection.get(
    where={"section": "candidate_summary"},
    include=["documents", "metadatas"],
)

policy_index = {}

for document, metadata in zip(
    policy_result["documents"],
    policy_result["metadatas"],
):
    policy_id = metadata.get("policy_id")
    title = str(metadata.get("title", "")).strip()

    if policy_id and title:
        policy_index[policy_id] = {
            "policy_id": policy_id,
            "title": title,
            "normalized_title": normalize_text(title),
            "document": document,
            "metadata": metadata,
        }


print("정책 색인 수:", len(policy_index))
print("Gemini 모델:", GEMINI_MODEL)

정책 색인 수: 1418
Gemini 모델: gemini-3.5-flash-lite


## 1. Gemini 질문 분류와 API 호출 제한 처리

In [4]:
import time
import re
import json

ALLOWED_SECTIONS = [
    "overview",
    "target_criteria",
    "benefit",
    "application",
    "documents",
    "contact_source",
    "administrative",
]

# 무료 등급 15 RPM보다 여유 있게 모든 Gemini 요청 사이를 5초 이상 띄운다.
MIN_GEMINI_INTERVAL_SECONDS = 5.0
_last_gemini_call_at = 0.0


def get_retry_wait_seconds(error, default=15.0):
    """Gemini 429 오류에 포함된 권장 대기시간을 읽는다."""
    message = str(error)
    patterns = [
        r"retry in ([0-9.]+)s",
        r"'retryDelay': '([0-9.]+)s'",
        r'"retryDelay": "([0-9.]+)s"',
    ]
    for pattern in patterns:
        match = re.search(pattern, message, flags=re.IGNORECASE)
        if match:
            return float(match.group(1)) + 2.0
    return default


def gemini_generate(contents, *, json_mode=False, max_retries=5):
    """모든 Gemini 호출에 공통 간격과 429 재시도를 적용한다."""
    global _last_gemini_call_at

    for attempt in range(max_retries):
        elapsed = time.monotonic() - _last_gemini_call_at
        wait_before_call = max(0.0, MIN_GEMINI_INTERVAL_SECONDS - elapsed)
        if wait_before_call:
            time.sleep(wait_before_call)

        try:
            config_kwargs = {"temperature": 0}
            if json_mode:
                config_kwargs["response_mime_type"] = "application/json"

            response = gemini.models.generate_content(
                model=GEMINI_MODEL,
                contents=contents,
                config=types.GenerateContentConfig(**config_kwargs),
            )
            _last_gemini_call_at = time.monotonic()
            return response

        except Exception as error:
            _last_gemini_call_at = time.monotonic()
            if attempt == max_retries - 1:
                raise

            wait_seconds = get_retry_wait_seconds(
                error,
                default=15.0 * (attempt + 1),
            )
            print(
                f"Gemini 호출 재시도 {attempt + 1}/{max_retries}: "
                f"{wait_seconds:.1f}초 대기"
            )
            time.sleep(wait_seconds)


def classify_question(question):
    """Gemini가 질문 의도와 조회할 DB 섹션을 결정한다."""
    prompt = f"""
당신은 복지정책 질문 분류기입니다.
사용자의 질문 목적을 분석하여 JSON만 반환하세요.

사용 가능한 intent:
- policy_overview: 정책 전반 설명
- eligibility: 지원 대상이나 선정 기준
- benefit: 지원 금액, 혜택, 서비스 내용
- application: 신청 방법이나 절차
- required_documents: 제출 서류 또는 공식 서식
- contact: 문의처
- administrative: 시행 기간, 수정일, 법적 근거

사용 가능한 section:
{ALLOWED_SECTIONS}

분류 규칙:
- 정책 전반 설명: overview, target_criteria, benefit
- 자격 또는 선정 기준: target_criteria
- 지원 금액·혜택·서비스: benefit
- 신청 방법 또는 절차: application
- 제출 서류·서식: documents, application
- 문의처: contact_source
- 시행 기간·수정일·법적 근거: administrative
- 두 가지를 함께 물으면 필요한 section을 모두 포함
- 질문에 없는 목적은 추가하지 말 것

반환 형식:
{{
  "intent": "대표 intent 하나",
  "sections": ["필요한 section"],
  "reason": "짧은 분류 이유"
}}

사용자 질문:
{question}
"""
    response = gemini_generate(prompt, json_mode=True)
    result = json.loads(response.text)
    sections = [
        section for section in result.get("sections", [])
        if section in ALLOWED_SECTIONS
    ]
    if not sections:
        sections = ["overview"]
    return {
        "intent": result.get("intent", "policy_overview"),
        "sections": sections,
        "reason": result.get("reason", ""),
    }


## 2. 벡터 검색 + 서비스 시작 시 받은 지역의 단계적 필터로 정책 후보 결정


In [5]:
def title_token_bonus(title, question):
    """질문에 포함된 정책명 핵심 단어 비율로 가산점을 계산한다."""
    generic_words = {"지원", "사업", "정책", "서비스", "관련", "및"}
    title_words = [
        word for word in re.findall(r"[가-힣A-Za-z0-9]+", str(title))
        if len(word) >= 2 and word not in generic_words
    ]
    if not title_words:
        return 0.0, []

    normalized_question = normalize_text(question)
    matched_words = [
        word for word in title_words
        if normalize_text(word) in normalized_question
    ]
    ratio = len(matched_words) / len(title_words)
    return 0.50 * ratio, matched_words


def find_exact_policy_mentions(question):
    normalized_question = normalize_text(question)
    return [
        policy for policy in policy_index.values()
        if policy["normalized_title"]
        and policy["normalized_title"] in normalized_question
    ]


# DB에 실제 저장된 공식 행정구역 이름을 기준으로 지역을 해석한다.
REGION_LEVEL1_VALUES = sorted({
    str(policy["metadata"].get("region_level1", "")).strip()
    for policy in policy_index.values()
    if str(policy["metadata"].get("region_level1", "")).strip()
}, key=len, reverse=True)


def parse_selected_region(region_text):
    """
    서비스 시작 시 선택받은 공식 지역 문자열을 시도/시군구로 나눈다.
    예: '부산광역시 해운대구' -> ('부산광역시', '해운대구')
    """
    text = str(region_text or "").strip()
    if not text or text in {"중앙부처", "전국"}:
        return None, None

    normalized = normalize_text(text)
    level1 = next(
        (
            value for value in REGION_LEVEL1_VALUES
            if normalize_text(value) in normalized
        ),
        None,
    )
    if not level1:
        raise ValueError(f"DB 공식 명칭과 일치하지 않는 시도입니다: {text}")

    remainder = normalized.replace(normalize_text(level1), "", 1)
    level2 = remainder or None
    return level1, level2


def make_where(*conditions):
    valid = [condition for condition in conditions if condition]
    if len(valid) == 1:
        return valid[0]
    return {"$and": valid}


def region_search_scopes(region_text):
    """
    다른 지자체 정책을 섞지 않는다.
    시군구가 있으면 같은 시군구 -> 같은 시도의 광역정책 -> 중앙부처 순서다.
    """
    level1, level2 = parse_selected_region(region_text)
    scopes = []

    if level1 and level2:
        scopes.append({
            "name": "district",
            "label": f"{level1} {level2}",
            "bonus": 0.60,
            "where": make_where(
                {"section": {"$eq": "candidate_summary"}},
                {"source": {"$eq": "local"}},
                {"region_level1": {"$eq": level1}},
                {"region_level2": {"$eq": level2}},
            ),
        })

        # region_level2가 빈 정책은 시도 전체에 적용되는 광역 정책으로 본다.
        scopes.append({
            "name": "province",
            "label": f"{level1} 광역",
            "bonus": 0.35,
            "where": make_where(
                {"section": {"$eq": "candidate_summary"}},
                {"source": {"$eq": "local"}},
                {"region_level1": {"$eq": level1}},
                {"region_level2": {"$eq": ""}},
            ),
        })
    elif level1:
        # 시도만 받은 경우에는 그 시도 안의 정책을 검색한다.
        scopes.append({
            "name": "province",
            "label": level1,
            "bonus": 0.35,
            "where": make_where(
                {"section": {"$eq": "candidate_summary"}},
                {"source": {"$eq": "local"}},
                {"region_level1": {"$eq": level1}},
            ),
        })

    # 중앙부처 정책은 거주 지역과 관계없이 받을 수 있으므로 함께 검색한다.
    scopes.append({
        "name": "national",
        "label": "중앙부처",
        "bonus": 0.10,
        "where": make_where(
            {"section": {"$eq": "candidate_summary"}},
            {"source": {"$eq": "national"}},
        ),
    })
    return level1, level2, scopes


def resolve_policy(question, region_text=None, n_results=30, return_top_k=5):
    """
    서비스에서 미리 받은 지역으로 검색 범위를 제한한 뒤 Qwen 벡터 검색과
    정책명 일치를 이용해 후보를 정렬한다. expected_policy_id는 사용하지 않는다.
    """
    query_vector = embedder.encode(
        question,
        prompt_name="query",
        normalize_embeddings=True,
        show_progress_bar=False,
    ).tolist()

    level1, level2, scopes = region_search_scopes(region_text)
    candidates = {}

    for scope in scopes:
        vector_result = collection.query(
            query_embeddings=[query_vector],
            n_results=n_results,
            where=scope["where"],
            include=["documents", "metadatas", "distances"],
        )

        for document, metadata, distance in zip(
            vector_result["documents"][0],
            vector_result["metadatas"][0],
            vector_result["distances"][0],
        ):
            policy_id = metadata.get("policy_id")
            if not policy_id:
                continue

            title = metadata.get("title", "")
            vector_score = 1.0 - float(distance)
            title_bonus, matched_words = title_token_bonus(title, question)
            score = vector_score + title_bonus + scope["bonus"]

            candidate = {
                "policy_id": policy_id,
                "title": title,
                "normalized_title": normalize_text(title),
                "document": document,
                "metadata": metadata,
                "distance": float(distance),
                "score": score,
                "region_match_level": scope["name"],
                "region_scope": scope["label"],
                "reasons": [
                    f"벡터 유사도 {vector_score:.3f}",
                    f"검색 범위: {scope['label']}",
                    *(
                        ["정책명 단어: " + ", ".join(matched_words)]
                        if matched_words else []
                    ),
                ],
            }
            previous = candidates.get(policy_id)
            if previous is None or candidate["score"] > previous["score"]:
                candidates[policy_id] = candidate

    # 정식 정책명이 통째로 들어가도 허용된 지역 범위의 후보만 우대한다.
    for exact in find_exact_policy_mentions(question):
        policy_id = exact["policy_id"]
        if policy_id in candidates:
            candidates[policy_id]["score"] += 1.0
            candidates[policy_id]["reasons"].append("정책명 전체 일치")

    ranked = sorted(
        candidates.values(),
        key=lambda item: (-item["score"], item["policy_id"]),
    )
    search_info = {
        "region_level1": level1,
        "region_level2": level2,
        "search_scopes": [scope["label"] for scope in scopes],
    }
    return (
        ranked[0] if ranked else None,
        ranked[:return_top_k],
        search_info,
    )


def show_policy_candidates(question, region_text=None, top_k=5):
    selected, candidates, search_info = resolve_policy(
        question,
        region_text=region_text,
        return_top_k=top_k,
    )
    print("질문:", question)
    print("검색 지역:", search_info)
    for rank, item in enumerate(candidates, start=1):
        print(
            f"{rank}. {item['title']} [{item['policy_id']}] "
            f"점수={item['score']:.3f} / "
            + " / ".join(item["reasons"])
        )
    return selected


## 3. 선택된 정책의 세부 원문 조회와 근거 기반 답변 생성

In [6]:
def get_policy_contexts(policy_id, sections):
    result = collection.get(
        where={
            "$and": [
                {"policy_id": {"$eq": policy_id}},
                {"section": {"$in": sections}},
            ]
        },
        include=["documents", "metadatas"],
    )

    records = []
    for chunk_id, document, metadata in zip(
        result["ids"], result["documents"], result["metadatas"]
    ):
        records.append({
            "chunk_id": chunk_id,
            "document": document,
            "section": metadata.get("section"),
            "policy_id": metadata.get("policy_id"),
            "title": metadata.get("title"),
        })

    section_order = {section: index for index, section in enumerate(sections)}
    records.sort(key=lambda item: (
        section_order.get(item["section"], 999),
        item["chunk_id"],
    ))
    return records


def generate_grounded_answer(question, contexts):
    if not contexts:
        return (
            "공식 자료에서 질문에 답할 "
            "근거를 찾지 못했습니다."
        )

    context_text = "\n\n".join(
        (
            f"[근거 {index} / {item['section']}]\n"
            f"{item['document']}"
        )
        for index, item in enumerate(
            contexts,
            start=1,
        )
    )

    prompt = f"""
당신은 공공 복지정책의 공식 원문을 정확하게 전달하는 답변 도우미입니다.

아래의 공식 정책 원문만 사용해 사용자의 질문에 답하세요.

답변 규칙:
1. 사용자가 물어본 내용에만 답하세요.
2. 공식 원문에 명시된 사실만 사용하세요.
3. 금액, 기간, 연령, 비율, 횟수, 지역은 원문 그대로 정확히 쓰세요.
4. 사용자가 말한 개인 조건을 근거 없이 자격 충족으로 판단하지 마세요.
5. 원문에 없는 주의사항이나 일반적인 안내 문구를 추가하지 마세요.
6. "실제 신청을 통해 확인하세요", "관련 기관에 문의하세요"와 같은 문구를 임의로 추가하지 마세요.
7. 같은 내용을 반복하지 마세요.
8. 간단한 질문은 1~2문장으로 답하세요.
9. 질문이 여러 내용을 요구한 경우에만 항목을 나누세요.
10. 정책명과 질문에 대한 핵심 답변을 포함하세요.
11. 사용자가 서식·첨부파일·온라인 신청 경로를 묻고 공식 원문에 URL이 있으면 해당 URL을 답변에 포함하세요.
12. 원문에 정보가 없을 때만 "공식 원문에서 확인되지 않습니다"라고 답하세요.

사용자 질문:
{question}

공식 정책 원문:
{context_text}

답변:
"""

    response = gemini_generate(
        prompt,
        json_mode=False,
    )

    return response.text.strip()

## 4. 전체 RAG 파이프라인 실행

In [7]:
import traceback


def run_rag_pipeline(source_df, limit=None):
    target_df = source_df.head(limit).copy() if limit is not None else source_df.copy()
    result_rows = []

    for number, (_, row) in enumerate(target_df.iterrows(), start=1):
        question = row["user_input"]
        result_row = row.to_dict()
        print(f"\n[{number}/{len(target_df)}] {row['id']} {question}")

        try:
            found_policy, top_candidates, search_info = resolve_policy(
                question, region_text=row.get("region")
            )
            if found_policy is None:
                raise ValueError("질문에 맞는 정책 후보를 찾지 못했습니다.")

            classification = classify_question(question)
            predicted_policy_id = found_policy["policy_id"]
            predicted_policy_name = found_policy["title"]
            predicted_intent = classification["intent"]
            predicted_sections = classification["sections"]

            retrieved_records = get_policy_contexts(
                predicted_policy_id, predicted_sections
            )
            reference_records = get_policy_contexts(
                row["expected_policy_id"], row["expected_sections"]
            )
            retrieved_contexts = [item["document"] for item in retrieved_records]
            retrieved_context_ids = [item["chunk_id"] for item in retrieved_records]
            reference_contexts = [item["document"] for item in reference_records]
            reference_context_ids = [item["chunk_id"] for item in reference_records]

            generated_answer = generate_grounded_answer(question, retrieved_records)

            expected_sections = set(row["expected_sections"])
            predicted_sections_set = set(predicted_sections)
            section_overlap = len(expected_sections & predicted_sections_set)
            retrieved_ids = set(retrieved_context_ids)
            reference_ids = set(reference_context_ids)
            id_overlap = len(retrieved_ids & reference_ids)

            result_row.update({
                "predicted_policy_id": predicted_policy_id,
                "predicted_policy_name": predicted_policy_name,
                "policy_candidates": [
                    {
                        "policy_id": item["policy_id"],
                        "title": item["title"],
                        "score": round(item["score"], 6),
                        "region_match_level": item["region_match_level"],
                        "region_scope": item["region_scope"],
                    }
                    for item in top_candidates
                ],
                "predicted_intent": predicted_intent,
                "predicted_sections": predicted_sections,
                "classification_reason": classification["reason"],
                "input_region_level1": search_info["region_level1"],
                "input_region_level2": search_info["region_level2"],
                "region_search_scopes": search_info["search_scopes"],
                "selected_region_match_level": found_policy["region_match_level"],
                "reference_contexts": reference_contexts,
                "reference_context_ids": reference_context_ids,
                "retrieved_contexts": retrieved_contexts,
                "retrieved_context_ids": retrieved_context_ids,
                "response": generated_answer,
                "policy_correct": int(predicted_policy_id == row["expected_policy_id"]),
                "intent_correct": int(predicted_intent == row["expected_intent"]),
                "section_exact_match": int(predicted_sections_set == expected_sections),
                "section_precision": (
                    section_overlap / len(predicted_sections_set)
                    if predicted_sections_set else 0.0
                ),
                "section_recall": (
                    section_overlap / len(expected_sections)
                    if expected_sections else 0.0
                ),
                "id_context_precision": (
                    id_overlap / len(retrieved_ids) if retrieved_ids else 0.0
                ),
                "id_context_recall": (
                    id_overlap / len(reference_ids) if reference_ids else 0.0
                ),
                "pipeline_error": None,
            })
            print("정책:", predicted_policy_name)
            print("의도/섹션:", predicted_intent, predicted_sections)
            print("답변:", generated_answer)

        except Exception as error:
            print("처리 실패:", error)
            traceback.print_exc()
            result_row.update({
                "predicted_policy_id": None,
                "predicted_policy_name": None,
                "policy_candidates": [],
                "predicted_intent": None,
                "predicted_sections": [],
                "classification_reason": None,
                "input_region_level1": None,
                "input_region_level2": None,
                "region_search_scopes": [],
                "selected_region_match_level": None,
                "reference_contexts": [],
                "reference_context_ids": [],
                "retrieved_contexts": [],
                "retrieved_context_ids": [],
                "response": None,
                "policy_correct": 0,
                "intent_correct": 0,
                "section_exact_match": 0,
                "section_precision": 0.0,
                "section_recall": 0.0,
                "id_context_precision": 0.0,
                "id_context_recall": 0.0,
                "pipeline_error": str(error),
            })

        result_rows.append(result_row)

    return pd.DataFrame(result_rows)


In [8]:
# 처음에는 3으로 시험하고, 정상 확인 후 None으로 바꿔 30문항을 실행한다.
PIPELINE_LIMIT = None

pipeline_df = run_rag_pipeline(eval_df, limit=PIPELINE_LIMIT)

display(pipeline_df[[
    "id",
    "expected_policy_name",
    "predicted_policy_name",
    "input_region_level1",
    "input_region_level2",
    "selected_region_match_level",
    "predicted_intent",
    "predicted_sections",
    "policy_correct",
    "intent_correct",
    "section_exact_match",
    "response",
    "pipeline_error",
]])



[1/30] RAGAS01 장애인보장구수리지원사업은 얼마 정도 지원받을 수 있나요?


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


정책: 장애인보장구수리지원사업
의도/섹션: benefit ['benefit']
답변: 장애인보장구수리지원사업의 지원 혜택은 휠체어 패드교체, 브레이크 수리 및 일반 부속품 등 수리비용을 받지 않고 서비스 하며, 단 부품비용은 예산 범위 내에서 1인당 20만원 범위 내에서 소득에 따라 차등 지원합니다.

[2/30] RAGAS02 국가보훈대상자가 받는 보훈영예수당은 얼마나 지급되나요?
정책: 보훈영예수당
의도/섹션: benefit ['benefit']
답변: 보훈영예수당은 월 25만원씩 현금으로 지급됩니다.

[3/30] RAGAS03 25살 직장인인데 청년 중개보수 및 이사비 지원을 받을 수 있는 기준이 무엇인가요?
정책: 부산 청년 중개보수 및 이사비 지원
의도/섹션: eligibility ['target_criteria']
답변: 부산 청년 중개보수 및 이사비 지원의 지원 대상 및 선정 기준은 연령 18~39세의 부산시 전입 또는 부산 시내 이사 후 전입신고를 한 청년, 기준 중위소득 120% 이하(월 2,871천원, 2025년 기준)인 일하는 청년, 임대차 거래금액 1.5억원 이하의 전월세 거주 주택 기준을 충족하며, 무주택자이자 주민등록상 청년 세대주 가구여야 합니다.

[4/30] RAGAS04 24살 미취업 청년입니다. 청년 구직활동비를 지원받으려면 어떤 자격을 갖춰야 하나요?
정책: 해운대구 청년 구직활동비 지원 사업
의도/섹션: eligibility ['target_criteria']
답변: 해운대구 청년 구직활동비 지원 사업의 지원 자격은 다음 조건을 모두 만족해야 합니다.

1. 공고일 기준 19세 이상 39세 이하 청년
2. 공고일 기준 해운대구 주민등록자 및 미취업 청년
3. 가구 기준중위소득 150% 이하 (건강보험료 기준)

[5/30] RAGAS05 임산부 태아기형아 검사 쿠폰 지원은 어떤 정책인가요?
정책: 임산부 태아기형아 검사 쿠폰 지원
의도/섹션: policy_overview ['overview', 'target_criteri

,id,expected_policy_name,predicted_policy_name,input_region_level1,input_region_level2,selected_region_match_level,predicted_intent,predicted_sections,policy_correct,intent_correct,section_exact_match,response,pipeline_error
0,RAGAS01,장애인보장구수리지원사업,장애인보장구수리지원사업,전북특별자치도,남원시,district,benefit,[benefit],1,1,1,"장애인보장구수리지원사업의 지원 혜택은 휠체어 패드교체, 브레이크 수리 및 일반 부속...",None
1,RAGAS02,보훈영예수당,보훈영예수당,강원특별자치도,인제군,district,benefit,[benefit],1,1,1,보훈영예수당은 월 25만원씩 현금으로 지급됩니다.,None
2,RAGAS03,부산 청년 중개보수 및 이사비 지원,부산 청년 중개보수 및 이사비 지원,부산광역시,NaN,province,eligibility,[target_criteria],1,1,1,부산 청년 중개보수 및 이사비 지원의 지원 대상 및 선정 기준은 연령 18~39세의...,None
3,RAGAS04,해운대구 청년 구직활동비 지원 사업,해운대구 청년 구직활동비 지원 사업,부산광역시,해운대구,district,eligibility,[target_criteria],1,1,1,해운대구 청년 구직활동비 지원 사업의 지원 자격은 다음 조건을 모두 만족해야 합니다...,None
4,RAGAS05,임산부 태아기형아 검사 쿠폰 지원,임산부 태아기형아 검사 쿠폰 지원,경상북도,김천시,district,policy_overview,"[overview, target_criteria, benefit]",1,1,1,'임산부 태아기형아 검사 쿠폰 지원'은 태아 기형아의 유무를 조기 발견 관리하고 출...,None
5,RAGAS06,신혼부부 및 청년 주택자금 대출이자 지원 사업,신혼부부 및 청년 주택자금 대출이자 지원 사업,전북특별자치도,정읍시,district,application,[application],1,1,1,'신혼부부 및 청년 주택자금 대출이자 지원 사업'의 신청 방법은 해당 주소지 읍/면...,None
6,RAGAS07,신혼부부 및 청년 주택자금 대출이자 지원 사업,신혼부부 및 청년 주택자금 대출이자 지원 사업,전북특별자치도,정읍시,district,benefit,"[benefit, administrative]",1,1,0,[신혼부부 및 청년 주택자금 대출이자 지원 사업] \n지원 금액은 신청일 기준 연 ...,None
7,RAGAS08,신혼부부 및 청년 주택자금 대출이자 지원 사업,신혼부부 및 청년 주택자금 대출이자 지원 사업,전북특별자치도,정읍시,district,eligibility,[target_criteria],1,1,1,신혼부부 및 청년 주택자금 대출이자 지원 사업의 청년 선정 기준은 정읍시에 6개월 ...,None
8,RAGAS09,장애인활동지원 시추가지원사업,장애인활동지원 시추가지원사업,경상남도,밀양시,district,policy_overview,"[overview, benefit]",1,1,0,장애인활동지원 시추가지원사업은 국비 장애인활동지원사업의 이용시간 부족으로 불편을 겪...,None
9,RAGAS10,장애인활동지원 시추가지원사업,장애인활동지원 시추가지원사업,경상남도,밀양시,district,benefit,[benefit],1,1,1,"장애인활동지원 시추가지원사업의 지원 혜택은 활동지원이며, 구체적인 지원 내용은 다음...",None


In [9]:
successful_df = pipeline_df[pipeline_df["pipeline_error"].isna()].copy()

print("전체 문항:", len(pipeline_df))
print("성공 문항:", len(successful_df))
print("실패 문항:", len(pipeline_df) - len(successful_df))
for column, label in [
    ("policy_correct", "정책 식별 정확도"),
    ("intent_correct", "의도 분류 정확도"),
    ("section_exact_match", "섹션 완전 일치율"),
    ("section_precision", "섹션 Precision"),
    ("section_recall", "섹션 Recall"),
    ("id_context_precision", "청크 ID Context Precision"),
    ("id_context_recall", "청크 ID Context Recall"),
]:
    print(f"{label}: {pipeline_df[column].mean():.4f}")


전체 문항: 30
성공 문항: 30
실패 문항: 0
정책 식별 정확도: 0.9333
의도 분류 정확도: 0.9667
섹션 완전 일치율: 0.8667
섹션 Precision: 0.9500
섹션 Recall: 0.9833
청크 ID Context Precision: 0.8833
청크 ID Context Recall: 0.9167


## 5. 파이프라인 결과 저장

In [10]:
OUTPUT_DIR = PROJECT_DIR / "data" / "evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PIPELINE_RESULT_FILE = OUTPUT_DIR / "paper_rag_pipeline_results.xlsx"
pipeline_save_df = pipeline_df.copy()

json_columns = [
    "expected_sections",
    "policy_candidates",
    "predicted_sections",
    "region_search_scopes",
    "reference_contexts",
    "reference_context_ids",
    "retrieved_contexts",
    "retrieved_context_ids",
]
for column in json_columns:
    if column in pipeline_save_df.columns:
        pipeline_save_df[column] = pipeline_save_df[column].apply(
            lambda value: json.dumps(value, ensure_ascii=False)
            if isinstance(value, (list, dict)) else value
        )

pipeline_save_df.to_excel(PIPELINE_RESULT_FILE, index=False)
print("저장 위치:", PIPELINE_RESULT_FILE)


저장 위치: C:\ai_project01\WelfareOne\data\evaluation\paper_rag_pipeline_results.xlsx


## 6. RAGAS 평가 준비

In [11]:
import sys
import types as python_types
import asyncio

vertex_module_name = (
    "langchain_community.chat_models.vertexai"
)

if vertex_module_name not in sys.modules:
    vertex_module = python_types.ModuleType(
        vertex_module_name
    )

    class ChatVertexAI:
        pass

    vertex_module.ChatVertexAI = ChatVertexAI
    sys.modules[vertex_module_name] = vertex_module


from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import (
    Faithfulness,
    ContextPrecisionWithReference,
    ContextRecall,
    FactualCorrectness,
)


# RAGAS의 비동기 ascore()에 맞는 비동기 클라이언트
ragas_openai_client = AsyncOpenAI(
    api_key=api_key,
    base_url=(
        "https://generativelanguage.googleapis.com/"
        "v1beta/openai/"
    ),
)


ragas_llm = llm_factory(
    model=GEMINI_MODEL,
    provider="openai",
    client=ragas_openai_client,
    adapter="instructor",
    temperature=0,
)


ragas_metrics = {
    "faithfulness": Faithfulness(
        llm=ragas_llm,
    ),
    "context_precision": (
        ContextPrecisionWithReference(
            llm=ragas_llm,
        )
    ),
    "context_recall": ContextRecall(
        llm=ragas_llm,
    ),
    "factual_correctness": (
        FactualCorrectness(
            llm=ragas_llm,
        )
    ),
}

print("RAGAS 비동기 평가 모델 준비 완료")
print("평가 모델:", GEMINI_MODEL)
print("클라이언트:", type(ragas_openai_client).__name__)

RAGAS 비동기 평가 모델 준비 완료
평가 모델: gemini-3.5-flash-lite
클라이언트: AsyncOpenAI


In [12]:
async def score_ragas_metric(metric, kwargs, max_retries=5):
    """RAGAS 내부 Gemini 호출도 429 권장시간에 맞춰 재시도한다."""
    for attempt in range(max_retries):
        try:
            result = await metric.ascore(**kwargs)
            await asyncio.sleep(MIN_GEMINI_INTERVAL_SECONDS)
            return result
        except Exception as error:
            if attempt == max_retries - 1:
                raise
            wait_seconds = get_retry_wait_seconds(
                error, default=20.0 * (attempt + 1)
            )
            print(
                f"RAGAS 호출 재시도 {attempt + 1}/{max_retries}: "
                f"{wait_seconds:.1f}초 대기"
            )
            await asyncio.sleep(wait_seconds)


RAGAS_CHECKPOINT_FILE = OUTPUT_DIR / "paper_ragas_checkpoint.csv"


def load_ragas_checkpoint():
    if not RAGAS_CHECKPOINT_FILE.exists():
        return pd.DataFrame()
    saved = pd.read_csv(RAGAS_CHECKPOINT_FILE)
    return saved.drop_duplicates(subset=["id"], keep="last")


def save_ragas_checkpoint(score_rows):
    pd.DataFrame(score_rows).to_csv(
        RAGAS_CHECKPOINT_FILE, index=False, encoding="utf-8-sig"
    )


async def run_ragas_evaluation(source_df, limit=3, resume=True):
    target_df = source_df.head(limit).copy() if limit is not None else source_df.copy()
    checkpoint_df = load_ragas_checkpoint() if resume else pd.DataFrame()
    score_rows = checkpoint_df.to_dict("records") if len(checkpoint_df) else []
    completed_ids = set(checkpoint_df["id"].astype(str)) if len(checkpoint_df) else set()

    for number, (_, row) in enumerate(target_df.iterrows(), start=1):
        row_id = str(row["id"])
        if row_id in completed_ids:
            print(f"RAGAS 평가 {number}/{len(target_df)}: {row_id} - 체크포인트 사용")
            continue

        print(f"\nRAGAS 평가 {number}/{len(target_df)}: {row_id}")
        score_row = {"id": row_id}
        try:
            contexts = row["retrieved_contexts"]
            if not isinstance(contexts, list) or not contexts:
                raise ValueError("retrieved_contexts가 비어 있습니다.")

            metric_inputs = {
                "faithfulness": {
                    "user_input": row["user_input"],
                    "response": row["response"],
                    "retrieved_contexts": contexts,
                },
                "context_precision": {
                    "user_input": row["user_input"],
                    "reference": row["reference"],
                    "retrieved_contexts": contexts,
                },
                "context_recall": {
                    "user_input": row["user_input"],
                    "reference": row["reference"],
                    "retrieved_contexts": contexts,
                },
                "factual_correctness": {
                    "response": row["response"],
                    "reference": row["reference"],
                },
            }

            for name, metric in ragas_metrics.items():
                result = await score_ragas_metric(metric, metric_inputs[name])
                score_row[name] = result.value
                score_row[f"{name}_reason"] = result.reason
                print(f"{name}: {result.value}")
            score_row["ragas_error"] = None

        except Exception as error:
            print("RAGAS 평가 실패:", error)
            for name in ragas_metrics:
                score_row[name] = None
                score_row[f"{name}_reason"] = None
            score_row["ragas_error"] = str(error)

        score_rows = [r for r in score_rows if str(r.get("id")) != row_id]
        score_rows.append(score_row)
        save_ragas_checkpoint(score_rows)
        completed_ids.add(row_id)

    result_df = pd.DataFrame(score_rows)
    target_ids = set(target_df["id"].astype(str))
    return result_df[result_df["id"].astype(str).isin(target_ids)].copy()


In [13]:
# 처음에는 3개로 시험하고, 정상 확인 후 None으로 변경합니다.
RAGAS_LIMIT = None

# 처음부터 다시 채점하려면 아래 두 줄의 주석을 한 번만 해제해 실행하세요.
# if RAGAS_CHECKPOINT_FILE.exists():
#     RAGAS_CHECKPOINT_FILE.unlink()

ragas_scores_df = await run_ragas_evaluation(
    pipeline_df, limit=RAGAS_LIMIT, resume=True
)
display(ragas_scores_df)



RAGAS 평가 1/30: RAGAS01
faithfulness: 1.0
context_precision: 0.9999999999
context_recall: 1.0
factual_correctness: 0.67

RAGAS 평가 2/30: RAGAS02
faithfulness: 1.0
context_precision: 0.9999999999
context_recall: 1.0
factual_correctness: 0.0

RAGAS 평가 3/30: RAGAS03
faithfulness: 1.0
context_precision: 0.9999999999
context_recall: 1.0
factual_correctness: 1.0

RAGAS 평가 4/30: RAGAS04
faithfulness: 1.0
context_precision: 0.9999999999
context_recall: 1.0
factual_correctness: 0.67

RAGAS 평가 5/30: RAGAS05
faithfulness: 1.0
context_precision: 0.8333333332916666
context_recall: 1.0
factual_correctness: 1.0

RAGAS 평가 6/30: RAGAS06
faithfulness: 1.0
context_precision: 0.9999999999
context_recall: 1.0
factual_correctness: 0.67

RAGAS 평가 7/30: RAGAS07
faithfulness: 1.0
context_precision: 0.9999999999
context_recall: 1.0
factual_correctness: 0.67

RAGAS 평가 8/30: RAGAS08
faithfulness: 1.0
context_precision: 0.99999999995
context_recall: 1.0
factual_correctness: 0.8

RAGAS 평가 9/30: RAGAS09
faithfulness:

,id,faithfulness,faithfulness_reason,context_precision,context_precision_reason,context_recall,context_recall_reason,factual_correctness,factual_correctness_reason,ragas_error
0,RAGAS01,1.0,None,1.000000,None,1.0,None,0.67,None,None
1,RAGAS02,1.0,None,1.000000,None,1.0,None,0.00,None,None
2,RAGAS03,1.0,None,1.000000,None,1.0,None,1.00,None,None
3,RAGAS04,1.0,None,1.000000,None,1.0,None,0.67,None,None
4,RAGAS05,1.0,None,0.833333,None,1.0,None,1.00,None,None
5,RAGAS06,1.0,None,1.000000,None,1.0,None,0.67,None,None
6,RAGAS07,1.0,None,1.000000,None,1.0,None,0.67,None,None
7,RAGAS08,1.0,None,1.000000,None,1.0,None,0.80,None,None
8,RAGAS09,1.0,None,1.000000,None,1.0,None,0.50,None,None
9,RAGAS10,1.0,None,1.000000,None,1.0,None,0.86,None,None


In [14]:
comparison_df = pipeline_df.merge(
    ragas_scores_df,
    on="id",
    how="left",
)

display(
    comparison_df[
        [
            "id",
            "user_input",
            "retrieved_contexts",
            "response",
            "reference",
            "faithfulness",
            "faithfulness_reason",
            "factual_correctness",
            "factual_correctness_reason",
        ]
    ]
)

,id,user_input,retrieved_contexts,response,reference,faithfulness,faithfulness_reason,factual_correctness,factual_correctness_reason
0,RAGAS01,장애인보장구수리지원사업은 얼마 정도 지원받을 수 있나요?,"[정책명: 장애인보장구수리지원사업\n지원 혜택:\n지원 내용: 휠체어 패드교체, 브...","장애인보장구수리지원사업의 지원 혜택은 휠체어 패드교체, 브레이크 수리 및 일반 부속...",예산 범위 내에서 부품비용을 1인당 연간 최대 20만 원까지 소득에 따라 차등 지원...,1.0,None,0.67,None
1,RAGAS02,국가보훈대상자가 받는 보훈영예수당은 얼마나 지급되나요?,[정책명: 보훈영예수당\n지원 혜택:\n지원 내용: 월 25만원씩 보훈영예수당 지급...,보훈영예수당은 월 25만원씩 현금으로 지급됩니다.,보훈영예수당은 월 25만 원씩 지급합니다.,1.0,None,0.00,None
2,RAGAS03,25살 직장인인데 청년 중개보수 및 이사비 지원을 받을 수 있는 기준이 무엇인가요?,[정책명: 부산 청년 중개보수 및 이사비 지원\n지원 대상 및 선정 조건:\n생애주...,부산 청년 중개보수 및 이사비 지원의 지원 대상 및 선정 기준은 연령 18~39세의...,18~39세로 부산시에 전입했거나 부산 시내에서 이사 후 전입신고한 일하는 청년이 ...,1.0,None,1.00,None
3,RAGAS04,24살 미취업 청년입니다. 청년 구직활동비를 지원받으려면 어떤 자격을 갖춰야 하나요?,[정책명: 해운대구 청년 구직활동비 지원 사업\n지원 대상 및 선정 조건:\n생애주...,해운대구 청년 구직활동비 지원 사업의 지원 자격은 다음 조건을 모두 만족해야 합니다...,공고일 기준 19세 이상 39세 이하이며 해운대구에 주민등록을 둔 미취업 청년이어야...,1.0,None,0.67,None
4,RAGAS05,임산부 태아기형아 검사 쿠폰 지원은 어떤 정책인가요?,[정책명: 임산부 태아기형아 검사 쿠폰 지원\n정책 개요:\n정책 개요: 태아 기형...,'임산부 태아기형아 검사 쿠폰 지원'은 태아 기형아의 유무를 조기 발견 관리하고 출...,태아 기형의 조기 발견과 출산가정의 경제적 부담 완화를 위한 정책입니다. 김천시보건...,1.0,None,1.00,None
5,RAGAS06,신혼부부 및 청년 주택자금 대출이자 지원 사업은 어떻게 신청하나요?,[정책명: 신혼부부 및 청년 주택자금 대출이자 지원 사업\n신청 방법:\n신청 방식...,'신혼부부 및 청년 주택자금 대출이자 지원 사업'의 신청 방법은 해당 주소지 읍/면...,주소지 읍·면·동 주민센터를 방문해 신청서를 제출하면 됩니다. 접수 후 담당 부서가...,1.0,None,0.67,None
6,RAGAS07,신혼부부 및 청년 주택자금 대출이자 지원 사업의 지원 금액과 기간을 알려주세요.,[정책명: 신혼부부 및 청년 주택자금 대출이자 지원 사업\n지원 혜택:\n지원 내용...,[신혼부부 및 청년 주택자금 대출이자 지원 사업] \n지원 금액은 신청일 기준 연 ...,"주택자금 대출 잔액의 2%를 연 1회, 최대 300만 원까지 지원하며 해당 요건을 ...",1.0,None,0.67,None
7,RAGAS08,6개월째 거주 중인 30살 청년이고 연소득은 5천만 원입니다. 신혼부부 및 청년 주...,[정책명: 신혼부부 및 청년 주택자금 대출이자 지원 사업\n지원 대상 및 선정 조건...,신혼부부 및 청년 주택자금 대출이자 지원 사업의 청년 선정 기준은 정읍시에 6개월 ...,청년은 정읍시에 6개월 이상 주민등록을 두고 거주하는 18~45세이며 연소득 6천만...,1.0,None,0.80,None
8,RAGAS09,장애인활동지원 시추가지원사업은 어떤 도움을 주는 정책인가요?,[정책명: 장애인활동지원 시추가지원사업\n정책 개요:\n정책 개요: 국비 장애인활동...,장애인활동지원 시추가지원사업은 국비 장애인활동지원사업의 이용시간 부족으로 불편을 겪...,국가 장애인활동지원만으로 시간이 부족한 이용자에게 활동지원 시간을 추가로 제공해 자...,1.0,None,0.50,None
9,RAGAS10,장애인활동지원 시추가지원사업에서 받을 수 있는 서비스는 무엇인가요?,[정책명: 장애인활동지원 시추가지원사업\n지원 혜택:\n지원 내용: 활동지원\n① ...,"장애인활동지원 시추가지원사업의 지원 혜택은 활동지원이며, 구체적인 지원 내용은 다음...","개인위생 관리, 신체기능 유지, 식사 도움과 실내 이동 같은 신체활동 지원, 청소·...",1.0,None,0.86,None


In [15]:
# 점수가 낮거나 오류가 발생한 문항을 우선 확인합니다.
review_preview = comparison_df[
    (comparison_df["faithfulness"].fillna(0) <= 0.5)
    | (comparison_df["factual_correctness"].fillna(0) <= 0.5)
    | comparison_df["ragas_error"].notna()
].copy()

display(review_preview[[
    "id", "user_input", "retrieved_contexts", "response", "reference",
    "faithfulness", "faithfulness_reason",
    "factual_correctness", "factual_correctness_reason", "ragas_error",
]])


,id,user_input,retrieved_contexts,response,reference,faithfulness,faithfulness_reason,factual_correctness,factual_correctness_reason,ragas_error
1,RAGAS02,국가보훈대상자가 받는 보훈영예수당은 얼마나 지급되나요?,[정책명: 보훈영예수당\n지원 혜택:\n지원 내용: 월 25만원씩 보훈영예수당 지급...,보훈영예수당은 월 25만원씩 현금으로 지급됩니다.,보훈영예수당은 월 25만 원씩 지급합니다.,1.0,None,0.0,None,None
8,RAGAS09,장애인활동지원 시추가지원사업은 어떤 도움을 주는 정책인가요?,[정책명: 장애인활동지원 시추가지원사업\n정책 개요:\n정책 개요: 국비 장애인활동...,장애인활동지원 시추가지원사업은 국비 장애인활동지원사업의 이용시간 부족으로 불편을 겪...,국가 장애인활동지원만으로 시간이 부족한 이용자에게 활동지원 시간을 추가로 제공해 자...,1.0,None,0.5,None,None
10,RAGAS11,장애인보장구 수리비 지원은 어떤 절차로 신청하나요?,[정책명: 장애인보장구 수리비 지원\n신청 방법:\n신청 방식: 방문\n신청 방법:...,장애인보장구 수리비 지원의 신청 방법은 대상자가 보장구 수리 후 읍면행정복지센터에 ...,보장구를 수리한 뒤 읍·면 행정복지센터에 신청합니다. 행정복지센터가 자격 기준을 확...,1.0,None,0.4,None,None
11,RAGAS12,장애인보장구 수리비 지원 대상은 누구인가요?,[정책명: 장애인보장구 수리비 지원\n지원 대상 및 선정 조건:\n생애주기: 영유아...,장애인보장구 수리비 지원 대상은 전동휠체어 및 전동스쿠터를 이용하는 관내 등록장애인...,영암군에 등록된 장애인 중 전동스쿠터 또는 전동휠체어를 이용하는 사람이 대상입니다.,1.0,None,0.0,None,None
13,RAGAS14,희망아이출생 산모산후관리비지원은 어떻게 신청하나요?,"[정책명: 희망아이출생 산모산후관리비지원\n신청 방법:\n신청 방식: 방문, 인터넷...",희망아이출생 산모산후관리비지원은 출산서비스 통합처리신청서 제출을 통해 방문 또는 인...,출산서비스 통합처리 신청서를 제출해 신청합니다.,1.0,None,0.0,None,None
14,RAGAS15,희망아이출생 산모산후관리비지원을 신청할 때 사용할 수 있는 서식은 무엇인가요?,[정책명: 희망아이출생 산모산후관리비지원\n필요 서류 및 첨부 자료:\n서식·준비 ...,희망아이출생 산모산후관리비지원을 신청할 때 사용할 수 있는 서식은 [(초안)광주광역...,공식 자료에 출산서비스 통합처리 신청서 서식이 제공되어 있습니다.,1.0,None,0.0,None,None
16,RAGAS17,화장장려금은 얼마나 지원되나요?,[정책명: 화장장려금 지원\n지원 혜택:\n지원 내용: 타지역 화장장을 청송군민이 ...,화장장려금 지원 혜택은 타지역 화장장을 청송군민이 이용할 경우 기 지역주민이 부담하...,"청송군민이 다른 지역 화장장을 이용한 경우, 그 화장장 사용료에서 해당 지역 주민이...",0.5,None,1.0,None,None
19,RAGAS20,고속도로통행료 할인카드 발급수수료 지원의 지원액과 문의처를 알려주세요.,[정책명: 고속도로통행료 할인카드 발급수수료 지원\n지원 혜택:\n지원 내용: 고속...,"고속도로통행료 할인카드 발급수수료 지원의 지원액은 고속도로통행료 발급 수수료 4,0...","할인카드 발급수수료 4,000원을 지원합니다. 문의처는 음성군청 복지정책과이며 전화...",1.0,None,0.0,None,None
20,RAGAS21,미취업 청년이 구직활동비를 지원받으면 얼마를 어떤 용도로 사용할 수 있나요?,[정책명: 해운대구 청년 구직활동비 지원 사업\n지원 혜택:\n지원 내용: 지출에 ...,해운대구 청년 구직활동비 지원 사업의 구체적인 지원 금액과 사용 용도는 공식 원문에...,총 180만 원을 월 30만 원씩 6개월간 전용 체크카드 포인트로 지원합니다. 학원...,1.0,None,0.0,None,None
22,RAGAS23,청년 대중교통비 지원을 이용하려면 교통카드를 어떻게 등록해야 하나요?,[정책명: 청년 면접 지원 꿈나래\n신청 방법:\n신청 방식: 인터넷\n신청 방법:...,공식 원문에서 확인되지 않습니다.,무상교통 홈페이지에서 회원가입한 뒤 전용카드를 신청하고 등록합니다. 6~23세 이용...,0.0,None,0.0,None,None


## 8. 자동평가 보존 + 이상 문항 수동검수 열 추가 + 최종 결과 저장


In [16]:
FINAL_RESULT_FILE = OUTPUT_DIR / "paper_rag_ragas_final_results.xlsx"
SUMMARY_RESULT_FILE = OUTPUT_DIR / "paper_rag_ragas_summary.xlsx"
final_df = pipeline_df.merge(ragas_scores_df, on="id", how="left")

manual_columns = [
    "manual_faithfulness",
    "manual_factual_correctness",
    "manual_review_reason",
    "reviewer",
]
for column in manual_columns:
    if column not in final_df.columns:
        final_df[column] = pd.NA

final_df["manual_review_required"] = (
    (final_df["faithfulness"].fillna(0) <= 0.5)
    | (final_df["factual_correctness"].fillna(0) <= 0.5)
    | (final_df["policy_correct"] == 0)
    | final_df["ragas_error"].notna()
)

print("[파이프라인 평가]")
pipeline_metrics = {
    "정책 식별 정확도": final_df["policy_correct"].mean(),
    "의도 분류 정확도": final_df["intent_correct"].mean(),
    "섹션 완전 일치율": final_df["section_exact_match"].mean(),
    "섹션 Precision": final_df["section_precision"].mean(),
    "섹션 Recall": final_df["section_recall"].mean(),
    "청크 ID Context Precision": final_df["id_context_precision"].mean(),
    "청크 ID Context Recall": final_df["id_context_recall"].mean(),
}
for label, value in pipeline_metrics.items():
    print(f"{label}: {value:.4f}")

print("\n[RAGAS 자동평가]")
ragas_summary = {}
for metric_name in ragas_metrics:
    value = final_df[metric_name].mean()
    ragas_summary[metric_name] = value
    print(f"{metric_name}: {value:.4f}")

metric_summary_df = pd.DataFrame(
    [{"평가 구분": "파이프라인", "지표": k, "평균": v} for k, v in pipeline_metrics.items()]
    + [{"평가 구분": "RAGAS", "지표": k, "평균": v} for k, v in ragas_summary.items()]
)

question_type_summary = final_df.groupby("question_type", dropna=False).agg(
    문항수=("id", "count"),
    정책정확도=("policy_correct", "mean"),
    의도정확도=("intent_correct", "mean"),
    섹션정확도=("section_exact_match", "mean"),
    faithfulness=("faithfulness", "mean"),
    factual_correctness=("factual_correctness", "mean"),
).reset_index()

intent_summary = final_df.groupby("expected_intent", dropna=False).agg(
    문항수=("id", "count"),
    정책정확도=("policy_correct", "mean"),
    섹션정확도=("section_exact_match", "mean"),
    faithfulness=("faithfulness", "mean"),
    factual_correctness=("factual_correctness", "mean"),
).reset_index()

final_save_df = final_df.copy()
for column in json_columns:
    if column in final_save_df.columns:
        final_save_df[column] = final_save_df[column].apply(
            lambda value: json.dumps(value, ensure_ascii=False)
            if isinstance(value, (list, dict)) else value
        )

final_save_df.to_excel(FINAL_RESULT_FILE, index=False)
with pd.ExcelWriter(SUMMARY_RESULT_FILE, engine="openpyxl") as writer:
    metric_summary_df.to_excel(writer, sheet_name="전체지표", index=False)
    question_type_summary.to_excel(writer, sheet_name="질문유형별", index=False)
    intent_summary.to_excel(writer, sheet_name="의도별", index=False)

print("\n최종 문항별 결과:", FINAL_RESULT_FILE)
print("논문용 요약 결과:", SUMMARY_RESULT_FILE)
print("수동검수 필요 문항:", final_df.loc[final_df["manual_review_required"], "id"].tolist())

display(metric_summary_df)
display(question_type_summary)
display(intent_summary)


[파이프라인 평가]
정책 식별 정확도: 0.9333
의도 분류 정확도: 0.9667
섹션 완전 일치율: 0.8667
섹션 Precision: 0.9500
섹션 Recall: 0.9833
청크 ID Context Precision: 0.8833
청크 ID Context Recall: 0.9167

[RAGAS 자동평가]
faithfulness: 0.9500
context_precision: 0.8944
context_recall: 0.9000
factual_correctness: 0.5180

최종 문항별 결과: C:\ai_project01\WelfareOne\data\evaluation\paper_rag_ragas_final_results.xlsx
논문용 요약 결과: C:\ai_project01\WelfareOne\data\evaluation\paper_rag_ragas_summary.xlsx
수동검수 필요 문항: ['RAGAS02', 'RAGAS09', 'RAGAS11', 'RAGAS12', 'RAGAS14', 'RAGAS15', 'RAGAS17', 'RAGAS20', 'RAGAS21', 'RAGAS23', 'RAGAS28', 'RAGAS30']


,평가 구분,지표,평균
0,파이프라인,정책 식별 정확도,0.933333
1,파이프라인,의도 분류 정확도,0.966667
2,파이프라인,섹션 완전 일치율,0.866667
3,파이프라인,섹션 Precision,0.950000
4,파이프라인,섹션 Recall,0.983333
5,파이프라인,청크 ID Context Precision,0.883333
6,파이프라인,청크 ID Context Recall,0.916667
7,RAGAS,faithfulness,0.950000
8,RAGAS,context_precision,0.894444
9,RAGAS,context_recall,0.900000


,question_type,문항수,정책정확도,의도정확도,섹션정확도,faithfulness,factual_correctness
0,간접형,12,0.833333,0.916667,0.916667,0.916667,0.480833
1,직접형,18,1.000000,1.000000,0.833333,0.972222,0.542778


,expected_intent,문항수,정책정확도,섹션정확도,faithfulness,factual_correctness
0,application,6,0.833333,1.000000,0.833333,0.456667
1,benefit,11,0.909091,0.909091,0.954545,0.551818
2,contact,1,1.000000,1.000000,1.000000,0.670000
3,eligibility,7,1.000000,1.000000,1.000000,0.555714
4,policy_overview,3,1.000000,0.333333,1.000000,0.500000
5,required_documents,2,1.000000,0.500000,1.000000,0.335000


## 논문 작성 시 결과 해석 원칙

- 정책·의도·섹션·청크 지표는 **파이프라인의 중간 단계 정확도**로 보고합니다.
- RAGAS 점수는 **검색 문맥과 생성 답변의 품질**로 보고합니다.
- RAGAS가 원문과 일치하는 답변에 비정상적으로 낮은 점수를 준 경우 자동점수를 삭제하지 않고 `manual_*` 열에 수동 검수 결과와 이유를 기록합니다.
- 시험 실행 3문항의 평균을 최종 성능으로 사용하지 않습니다. 반드시 전체 30문항 실행 결과를 사용합니다.


## 9. 수동검수 반영 최종 집계

이 부분은 Gemini API나 RAGAS를 다시 호출하지 않습니다. 수동검수 완료 결과 파일을 읽어 자동점수와 최종점수를 함께 계산합니다. **수동검수 엑셀을 저장하고 닫은 뒤 아래 셀만 실행**하면 됩니다.


In [17]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path(r"C:\ai_project01\WelfareOne\data\evaluation")

MANUAL_REVIEW_FILE = OUTPUT_DIR / "paper_rag_ragas_final_results_manual_reviewed.xlsx"
MANUAL_SUMMARY_FILE = OUTPUT_DIR / "paper_rag_ragas_summary_manual_reviewed.xlsx"

if not MANUAL_REVIEW_FILE.exists():
    raise FileNotFoundError(
        f"수동검수 완료 파일을 찾을 수 없습니다: {MANUAL_REVIEW_FILE}"
    )

reviewed_df = pd.read_excel(MANUAL_REVIEW_FILE)
required_manual_columns = [
    "manual_faithfulness",
    "manual_factual_correctness",
    "manual_review_reason",
    "reviewer",
]
missing_manual_columns = [
    column for column in required_manual_columns
    if column not in reviewed_df.columns
]
if missing_manual_columns:
    raise ValueError(f"수동검수 열 누락: {missing_manual_columns}")

reviewed_df["final_faithfulness"] = pd.to_numeric(
    reviewed_df["manual_faithfulness"], errors="coerce"
).fillna(reviewed_df["faithfulness"])
reviewed_df["final_factual_correctness"] = pd.to_numeric(
    reviewed_df["manual_factual_correctness"], errors="coerce"
).fillna(reviewed_df["factual_correctness"])

manual_count = reviewed_df["manual_factual_correctness"].notna().sum()
print("전체 문항:", len(reviewed_df))
print("수동검수 반영 문항:", manual_count)
print("검수자:", reviewed_df.loc[reviewed_df["reviewer"].notna(), "reviewer"].unique().tolist())

score_comparison_df = pd.DataFrame([
    {
        "지표": "Faithfulness",
        "자동평가": reviewed_df["faithfulness"].mean(),
        "수동검수 반영": reviewed_df["final_faithfulness"].mean(),
    },
    {
        "지표": "Factual Correctness",
        "자동평가": reviewed_df["factual_correctness"].mean(),
        "수동검수 반영": reviewed_df["final_factual_correctness"].mean(),
    },
])

# 실제 서비스에서는 후보 목록에서 사용자가 정책을 선택하므로,
# 정책 검색이 맞았던 문항의 선택 이후 답변 품질도 보조지표로 확인한다.
selected_policy_df = reviewed_df[reviewed_df["policy_correct"] == 1].copy()
post_selection_df = pd.DataFrame([
    {
        "평가 범위": "전체 30문항",
        "문항 수": len(reviewed_df),
        "최종 Faithfulness": reviewed_df["final_faithfulness"].mean(),
        "최종 Factual Correctness": reviewed_df["final_factual_correctness"].mean(),
    },
    {
        "평가 범위": "정책 검색 성공 문항",
        "문항 수": len(selected_policy_df),
        "최종 Faithfulness": selected_policy_df["final_faithfulness"].mean(),
        "최종 Factual Correctness": selected_policy_df["final_factual_correctness"].mean(),
    },
])

question_type_final_df = reviewed_df.groupby(
    "question_type", dropna=False
).agg(
    문항수=("id", "count"),
    정책정확도=("policy_correct", "mean"),
    최종_Faithfulness=("final_faithfulness", "mean"),
    최종_Factual_Correctness=("final_factual_correctness", "mean"),
).reset_index()

intent_final_df = reviewed_df.groupby(
    "expected_intent", dropna=False
).agg(
    문항수=("id", "count"),
    정책정확도=("policy_correct", "mean"),
    최종_Faithfulness=("final_faithfulness", "mean"),
    최종_Factual_Correctness=("final_factual_correctness", "mean"),
).reset_index()

review_detail_df = reviewed_df.loc[
    reviewed_df["manual_factual_correctness"].notna(),
    [
        "id", "user_input", "faithfulness", "factual_correctness",
        "manual_faithfulness", "manual_factual_correctness",
        "final_faithfulness", "final_factual_correctness",
        "manual_review_reason", "reviewer",
    ],
].copy()

with pd.ExcelWriter(MANUAL_SUMMARY_FILE, engine="openpyxl") as writer:
    score_comparison_df.to_excel(writer, sheet_name="자동_수동비교", index=False)
    post_selection_df.to_excel(writer, sheet_name="서비스흐름기준", index=False)
    question_type_final_df.to_excel(writer, sheet_name="질문유형별", index=False)
    intent_final_df.to_excel(writer, sheet_name="의도별", index=False)
    review_detail_df.to_excel(writer, sheet_name="수동검수내역", index=False)

print("\n[자동평가와 수동검수 반영 비교]")
display(score_comparison_df.style.format({"자동평가": "{:.4f}", "수동검수 반영": "{:.4f}"}))
print("\n[실제 서비스 흐름 기준 보조 분석]")
display(post_selection_df.style.format({
    "최종 Faithfulness": "{:.4f}",
    "최종 Factual Correctness": "{:.4f}",
}))
print("\n[질문 유형별 최종 결과]")
display(question_type_final_df)
print("\n[수동검수 내역]")
display(review_detail_df)
print("\n수동검수 반영 요약 저장 위치:", MANUAL_SUMMARY_FILE)
print(
    "주의: reviewer가 'Codex 수동검수'로 되어 있습니다. "
    "논문에 사용하기 전 팀원이 근거를 확인하고 실제 검수자 이름으로 변경하세요."
)


전체 문항: 30
수동검수 반영 문항: 12
검수자: []

[자동평가와 수동검수 반영 비교]


,지표,자동평가,수동검수 반영
0,Faithfulness,0.9500,1.0000
1,Factual Correctness,0.5180,0.7647



[실제 서비스 흐름 기준 보조 분석]


,평가 범위,문항 수,최종 Faithfulness,최종 Factual Correctness
0,전체 30문항,30,1.0000,0.7647
1,정책 검색 성공 문항,28,1.0000,0.8193



[질문 유형별 최종 결과]


,question_type,문항수,정책정확도,최종_Faithfulness,최종_Factual_Correctness
0,간접형,12,0.833333,1.0,0.647500
1,직접형,18,1.000000,1.0,0.842778



[수동검수 내역]


,id,user_input,faithfulness,factual_correctness,manual_faithfulness,manual_factual_correctness,final_faithfulness,final_factual_correctness,manual_review_reason,reviewer
1,RAGAS02,국가보훈대상자가 받는 보훈영예수당은 얼마나 지급되나요?,1.0,0.0,1.0,1.0,1.0,1.0,월 25만 원 지급과 현금 지급 방식이 검색 원문 및 기준 답변과 일치한다. 자동 ...,NaN
8,RAGAS09,장애인활동지원 시추가지원사업은 어떤 도움을 주는 정책인가요?,1.0,0.5,1.0,1.0,1.0,1.0,정책 목적과 활동지원 서비스 내용이 모두 검색 원문에 근거한다. 기준 답변보다 상세...,NaN
10,RAGAS11,장애인보장구 수리비 지원은 어떤 절차로 신청하나요?,1.0,0.4,1.0,0.8,1.0,0.8,신청·서류 확인·송금 절차는 원문과 일치한다. 다만 생성 답변에서 지급계좌의 수취인...,NaN
11,RAGAS12,장애인보장구 수리비 지원 대상은 누구인가요?,1.0,0.0,1.0,1.0,1.0,1.0,‘관내 등록장애인’은 현재 정책 지역인 영암군 등록장애인을 뜻하며 전동휠체어·전동스...,NaN
13,RAGAS14,희망아이출생 산모산후관리비지원은 어떻게 신청하나요?,1.0,0.0,1.0,1.0,1.0,1.0,출산서비스 통합처리 신청서 제출과 방문·인터넷 신청 방식이 검색 원문에 근거하며 질...,NaN
14,RAGAS15,희망아이출생 산모산후관리비지원을 신청할 때 사용할 수 있는 서식은 무엇인가요?,1.0,0.0,1.0,0.5,1.0,0.5,신청서 파일과 URL은 원문에 있으나 조례안까지 신청에 사용할 서식으로 분류했다. ...,NaN
16,RAGAS17,화장장려금은 얼마나 지원되나요?,0.5,1.0,1.0,1.0,1.0,1.0,타지역 화장장 이용 시 지역주민 부담액을 제외한 차액의 50% 지원이라는 산정 방식...,NaN
19,RAGAS20,고속도로통행료 할인카드 발급수수료 지원의 지원액과 문의처를 알려주세요.,1.0,0.0,1.0,1.0,1.0,1.0,"지원액 4,000원, 담당 기관, 전화번호가 검색 원문 및 기준 답변과 모두 일치한...",NaN
20,RAGAS21,미취업 청년이 구직활동비를 지원받으면 얼마를 어떤 용도로 사용할 수 있나요?,1.0,0.0,1.0,0.0,1.0,0.0,"검색된 다른 정책의 원문 범위에서는 근거 없는 내용을 만들지 않았으나, 기준 정책이...",NaN
22,RAGAS23,청년 대중교통비 지원을 이용하려면 교통카드를 어떻게 등록해야 하나요?,0.0,0.0,1.0,0.0,1.0,0.0,"잘못 선택된 정책 원문에 해당 정보가 없다는 답변 자체는 문맥에 충실하지만, 기준 ...",NaN



수동검수 반영 요약 저장 위치: C:\ai_project01\WelfareOne\data\evaluation\paper_rag_ragas_summary_manual_reviewed.xlsx
주의: reviewer가 'Codex 수동검수'로 되어 있습니다. 논문에 사용하기 전 팀원이 근거를 확인하고 실제 검수자 이름으로 변경하세요.
